Load packages

In [17]:
import rasterio
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import scipy
from scipy.stats import pearsonr
import rioxarray
import glob
import pandas as pd

Find out which bands to use for every image

In [2]:
def extract_image_info(filename):
    """
    Extracts information from a TIF-file. More precisely, the name of the file, the number of bands,
    the spatial resolution, the CRS and the data types of the bands, as well as which band is which.

    Parameters
    ----------
    filename : string
        The file you want to look into
    
    Returns
    -------
    nothing (it just prints stuff)

    Example
    -------
    >>> extract_image_info("LandsatComposite_Zurich_1985.tif")
    LandsatComposite_Zurich_1985.tif
    number of bands: 7
    resolution:     (30.0, 30.0)
    CRS:           EPSG:32632
    data type:      ('float64', 'float64', 'float64', 'float64', 'float64', 'float64', 'float64')
    Band 1: Blue
    Band 2: Green
    Band 3: Red
    Band 4: NIR
    Band 5: SWIR1
    Band 6: SWIR2
    Band 7: TIR1
    """
    with rasterio.open(f"../data/raw/{filename}") as src:
        print(filename)
        print(f"number of bands: {src.count}")
        print(f"resolution: {src.res}")
        print(f"CRS: {src.crs}")
        print(f"data type: {src.dtypes}")

        for band, desc in enumerate(src.descriptions, 1):
            print(f"Band {band}: {desc}")
    return

In [3]:
for image in range(1985,2027,3):
    extract_image_info(f"LandsatComposite_Zurich_{image}.tif")

LandsatComposite_Zurich_1985.tif
number of bands: 7
resolution: (30.0, 30.0)
CRS: EPSG:32632
data type: ('float64', 'float64', 'float64', 'float64', 'float64', 'float64', 'float64')
Band 1: Blue
Band 2: Green
Band 3: Red
Band 4: NIR
Band 5: SWIR1
Band 6: SWIR2
Band 7: TIR1
LandsatComposite_Zurich_1988.tif
number of bands: 7
resolution: (30.0, 30.0)
CRS: EPSG:32632
data type: ('float64', 'float64', 'float64', 'float64', 'float64', 'float64', 'float64')
Band 1: Blue
Band 2: Green
Band 3: Red
Band 4: NIR
Band 5: SWIR1
Band 6: SWIR2
Band 7: TIR1
LandsatComposite_Zurich_1991.tif
number of bands: 7
resolution: (30.0, 30.0)
CRS: EPSG:32632
data type: ('float64', 'float64', 'float64', 'float64', 'float64', 'float64', 'float64')
Band 1: Blue
Band 2: Green
Band 3: Red
Band 4: NIR
Band 5: SWIR1
Band 6: SWIR2
Band 7: TIR1
LandsatComposite_Zurich_1994.tif
number of bands: 7
resolution: (30.0, 30.0)
CRS: EPSG:32632
data type: ('float64', 'float64', 'float64', 'float64', 'float64', 'float64', 'float6

Great, they all have the same CRS and resolution!

For NDVI, Band 3 and 4 are needed (NDVI = (NIR - Red) / (NIR + Red)).

For LST, Band 7 is needed.

Do I need to convert the values of Band 7 to get temperature?

In [4]:
def analyze_band_7(filename):
    """
    Analyzes band 7 of a TIF image, that means that it prints the minimum and maximum value of band 7, ignoring any potential NaN values.
    Assumes thermal band (TIR1) as band 7.

    Parameters
    ----------
    filename : string
        The file you want to look into
    
    Returns
    -------
    nothing (it just prints stuff)

    Example
    -------
    >>> analyze_band_7("LandsatComposite_Zurich_1985.tif")
    LandsatComposite_Zurich_1985.tif
    Band 7: min=292.0, max=318.9
    """
    with rasterio.open(f"../data/raw/{filename}") as src:
        band = src.read(7).astype(float)
        print(filename)
        print(f"Band {7}: min={np.nanmin(band):.1f}, max={np.nanmax(band):.1f}") # ignoring potential NaN values
    return

In [5]:
for image in range(1985,2027,3):
    analyze_band_7(f"LandsatComposite_Zurich_{image}.tif")

LandsatComposite_Zurich_1985.tif
Band 7: min=292.0, max=318.9
LandsatComposite_Zurich_1988.tif
Band 7: min=289.4, max=317.9
LandsatComposite_Zurich_1991.tif
Band 7: min=292.5, max=317.8
LandsatComposite_Zurich_1994.tif
Band 7: min=293.9, max=321.8
LandsatComposite_Zurich_1997.tif
Band 7: min=289.2, max=315.7
LandsatComposite_Zurich_2000.tif
Band 7: min=294.3, max=326.2
LandsatComposite_Zurich_2003.tif
Band 7: min=295.3, max=324.1
LandsatComposite_Zurich_2006.tif
Band 7: min=294.3, max=326.5
LandsatComposite_Zurich_2009.tif
Band 7: min=294.7, max=326.1
LandsatComposite_Zurich_2012.tif
Band 7: min=295.7, max=324.5
LandsatComposite_Zurich_2015.tif
Band 7: min=296.4, max=326.7
LandsatComposite_Zurich_2018.tif
Band 7: min=295.7, max=323.0
LandsatComposite_Zurich_2021.tif
Band 7: min=297.8, max=323.4
LandsatComposite_Zurich_2024.tif
Band 7: min=296.2, max=317.9


It's all already in Kelvin, so that's great!

In [6]:
files = sorted(glob.glob("../data/raw/*.tif"))
years = list(range(1985,2027,3))

scenes = []
for f, year in zip(files, years):
    ds = xr.open_dataset(f, engine="rasterio")

    # select the right bands
    red = ds["band_data"].sel(band=3)
    nir = ds["band_data"].sel(band=4)
    tir = ds["band_data"].sel(band=7)

    ndvi = (nir - red) / (nir + red) # calculate NDVI
    lst = tir - 273.15 # convert LST from Kelvin to Celsius

    scene = xr.Dataset({"NDVI": ndvi, "LST": lst})
    scene = scene.assign_coords(year=year)
    scenes.append(scene)

ds = xr.concat(scenes, dim="year")
print(ds.LST)

<xarray.DataArray 'LST' (year: 14, y: 252, x: 297)> Size: 8MB
array([[[23.87761016, 24.22112117, 24.52361594, ..., 24.43474742,
         24.41082128, 24.1749779 ],
        [22.91714654, 23.24014943, 23.3717432 , ..., 24.49285376,
         24.38689514, 24.12028958],
        [22.29335789, 22.62832385, 22.27114076, ..., 25.3200146 ,
         24.62273852, 24.29973563],
        ...,
        [28.26805685, 28.6525841 , 27.51951047, ..., 29.87965328,
         29.30542592, 29.25757364],
        [28.00999634, 28.16722526, 28.01341436, ..., 30.08131646,
         29.4045485 , 29.32935206],
        [27.75706286, 27.8835296 , 27.85618544, ..., 30.20436518,
         29.53101524, 29.38062236]],

       [[20.2442549 , 20.18273054, 19.96397726, ..., 25.07049914,
         25.05340904, 24.33904286],
        [19.61533922, 19.6119212 , 19.9708133 , ..., 25.05682706,
         25.04999102, 24.362969  ],
        [19.99815746, 19.25986514, 19.44785624, ..., 23.52042707,
         23.45035766, 23.00772407],
...
 

So, now I have NDVI and LST per pixel per image. This means I can now analyze everything.

First, I want to calculate whether there is any correlation between land surface temperature and vegetation, so, correlation between LST and NDVI.

In [7]:
correlation_NDVI_LST = xr.corr(ds["NDVI"], ds["LST"], dim="year")

In [8]:
fig, ax = plt.subplots(figsize=(16,9))

correlation_NDVI_LST.plot(
    ax=ax,
    cmap="RdBu",
    vmin=-1,
    vmax=1,
    cbar_kwargs={"label": "Pearson's R (NDVI vs. LST)"}
)

ax.set_title("Correlation NDVI vs. LST, Zürich 1985-2024 (tri-annual)")
plt.tight_layout()
plt.savefig("../outputs/correlation_NDVI_LST.png", dpi=300)
plt.close()

That equates to exactly nothing.

What I'm actually interested in is not the within-pixel correlation of LST and NDVI over the years, but the overall correlation of LST and NDVI over all pixel over the years. The same logic applies.

In [9]:
correlation_NDVI_LST_overall = xr.corr(ds["NDVI"], ds["LST"], dim=["x","y"])
correlation_NDVI_LST_overall

<xarray.DataArray (year: 14)> Size: 112B
array([-0.798212  , -0.69177295, -0.80443213, -0.78342883, -0.7832679 ,
       -0.80124398, -0.81180184, -0.76451434, -0.80767912, -0.7913617 ,
       -0.79583615, -0.79890885, -0.79105992, -0.78716256])
Coordinates:
  * year         (year) int64 112B 1985 1988 1991 1994 ... 2015 2018 2021 2024
    spatial_ref  int64 8B 0
    band         int64 8B 7
Attributes:
    AREA_OR_POINT:  Area
    long_name:      ('Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2', 'TIR1')

Negative correlation, very good! More vegetation equals lower temperature! Now plot that.

In [10]:
fig, ax = plt.subplots(figsize=(16, 9))

correlation_NDVI_LST_overall.plot(ax=ax, marker="o", color="blue", linewidth=2)

ax.set_ylim(-1, 0)
ax.set_title("Spatial correlation NDVI vs. LST per year, Zürich 1985-2024 (triannual)", fontsize=18)
ax.set_xlabel("Year", fontsize=15)
ax.set_ylabel("Pearson's R", fontsize=15)
ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("../outputs/spatial_correlation.png", dpi=300)
plt.close()

But do areas with higher NDVI (=more vegetation) heat up slower than built-up areas?

--> How did LST change over time and is it different in pixels with higher NDVI? --> Linear regression per pixel

In [11]:
LST_trend = ds["LST"].polyfit(dim="year", deg=1) # for each pixel a straight (deg=1) line over all years

LST_gradient = LST_trend["polyfit_coefficients"].sel(degree=1) # I want to extract the gradient, not the intercept

LST_gradient

<xarray.DataArray 'polyfit_coefficients' (y: 252, x: 297)> Size: 599kB
array([[0.1622971 , 0.17080209, 0.17635356, ..., 0.17008468, 0.14304852,
        0.13643784],
       [0.16858351, 0.16698843, 0.17437912, ..., 0.16626226, 0.13689107,
        0.13219349],
       [0.15793256, 0.16357166, 0.16943989, ..., 0.17682557, 0.17007216,
        0.15180516],
       ...,
       [0.20607531, 0.19939452, 0.20315058, ..., 0.09081792, 0.10400046,
        0.11258307],
       [0.20449651, 0.20057392, 0.19325836, ..., 0.08930047, 0.10179314,
        0.11226505],
       [0.20515507, 0.21033844, 0.20565463, ..., 0.09216384, 0.11240153,
        0.11463263]], shape=(252, 297))
Coordinates:
  * y        (y) float64 2kB 5.253e+06 5.253e+06 ... 5.245e+06 5.245e+06
  * x        (x) float64 2kB 4.6e+05 4.6e+05 4.6e+05 ... 4.688e+05 4.689e+05
    degree   int64 8B 1

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(16, 9))

LST_gradient.plot(ax=axes[0], cmap="RdBu_r", center=0, cbar_kwargs={"label": "°C per year"})
axes[0].set_title("LST trend per pixel")

NDVI_mean = ds["NDVI"].mean(dim="year") # calculate mean NDVI to see where vegetation is
NDVI_mean.plot(ax=axes[1], cmap="Greens", cbar_kwargs={"label": "NDVI"})
axes[1].set_title("mean NDVI (1985-2024)")

plt.tight_layout()
plt.savefig("../outputs/LST_trend_per_pixel.png", dpi=300)
plt.close()

That's okay but not yet very visible correlations --> remove top and bottom 10% of LST_trend and NDVI_mean

In [13]:
def clip_percentile(da, low=10, high=90):
    """
    Clips input data array at low (default 10th percentile) and high (default 90 percentile)

    Parameters
    ----------
    da: xarray.DataArray
        The data array you want to clip
    low: numeric
        The lower boundary
    high: numeric
        The upper boundary
    
    Returns
    -------
    xarray.DataArray where the lower and upper specified percent of the original xarray.DataArray are clipped.

    Example
    -------
    >>> clip_percentile(NDVI_mean)
    array([[0.83267414, 0.83950845,        nan, ..., 0.55465284, 0.65810631, 0.74341382],
        [       nan,        nan,        nan, ..., 0.52137461, 0.57735498, 0.7502168 ],
        ...,
        [0.53181509, 0.60708441, 0.63088431, ..., 0.66942733,        nan, 0.67238843],
        [0.5330407 , 0.60384443, 0.66501119, ..., 0.60584457, 0.58811869, 0.6049107]],
        shape=(252, 297)),
    Coordinates:
    * y            (y) float64 2kB 5.253e+06 5.253e+06 ... 5.245e+06 5.245e+06
    * x            (x) float64 2kB 4.6e+05 4.6e+05 4.6e+05 ... 4.688e+05 4.689e+05
        spatial_ref  int64 8B 0
        band         int64 8B 7
    Attributes:
        AREA_OR_POINT:  Area
        long_name:      ('Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2', 'TIR1'),
    np.float64(0.1734316215108281),
    np.float64(0.8789213236052438))
    """
    vmin = np.nanpercentile(da.values, low)
    vmax = np.nanpercentile(da.values, high)
    return da.where((da > vmin) & (da < vmax)), vmin, vmax

In [14]:
LST_gradient_clip, vmin_LST, vmax_LST = clip_percentile(LST_gradient, low=2, high=98)
NDVI_mean_clip, vmin_NDVI, vmax_NDVI = clip_percentile(NDVI_mean, low=5, high=100)

In [15]:
cmap_reds = plt.cm.Reds.copy()
cmap_reds.set_bad("blue")  # NaN = blue

cmap_greens = plt.cm.Greens.copy()
cmap_greens.set_bad("black") # NaN = black

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

LST_gradient_clip.plot(ax=axes[0], cmap=cmap_reds, center=0, vmin=vmin_LST, vmax=vmax_LST, cbar_kwargs={"label": "°C per year"})
axes[0].set_title("LST trend per pixel (2nd-98th percentiles)")

NDVI_mean_clip.plot(ax=axes[1], cmap=cmap_greens, vmin=vmin_NDVI, vmax= vmax_NDVI, cbar_kwargs={"label": "mean NDVI (1985-2024)"})
axes[1].set_title("mean NDVI (5th-100th percentiles)")

plt.tight_layout()
plt.savefig("../outputs/LST_trend_per_pixel_better.png")
plt.close()

Visually, it looks as if higher NDVI is in areas with smaller temperature increase. Is this also the case statistically?

In [26]:
# 2D --> 1D
trend_flat = LST_gradient_clip.values.flatten()
ndvi_flat  = NDVI_mean_clip.values.flatten()

# remove NaN
mask = ~(np.isnan(trend_flat) | np.isnan(ndvi_flat))

r, p = pearsonr(ndvi_flat[mask], trend_flat[mask])
print(f"r = {r}, p = {p}")

r = -0.10442513135651404, p = 3.4421629276196803e-165
